# MiniMind on Colab（T4 适配版）

仓库：`Ezra-Maker-MAX/minimind`（fork 自 `jingyaogong/minimind`，64M 参数）

> ⚠️ 本 notebook 的三处硬约束（Python 3.13 wheel / T4 无 bf16 / checkpoint 路径硬编码）
> 均通过**读源码核对**确认，但**尚未在真实 Colab 实例上端到端跑通**。首个 cell 的环境体检可自证。

本 notebook 规避的坑：

| # | 坑 | 处理 |
|---|---|---|
| 1 | `requirements.txt` 钉死 `numpy==1.26.4`，Colab 是 Python 3.13，该版本**无 cp313 wheel**，且 pip 回溯会占死 kernel | 不装整份 requirements，只装核对过的必需子集 |
| 2 | 训练脚本默认 `--dtype bfloat16`，T4（sm_75）**无 bf16 硬件支持** | 按 GPU 架构自动切 `float16` |
| 3 | 检查点路径 `../checkpoints` 在源码里**硬编码**（8 个训练脚本全部如此），`--save_dir` 管不到 | 项目整体放 Google Drive，天然持久化 |
| 4 | 免费层会话会被回收，mini 数据在 T4 上单次跑不完 | `--from_resume 1` 断点续训 |
| 5 | 免费 Drive 只有 15GB，完整版数据 22.4GB **装不下** | 用 mini（2.98GB）或 P2/P3（236MB） |

**路线建议**：P0 冒烟 → P2 对齐 / P3 LoRA（单次会话可完成）→ P1 基座（长线）

> 前置：菜单「修改 → 笔记本设置 → 硬件加速器」选 **T4 GPU** 并保存。

In [ ]:
# ── 0. 环境体检（决定 dtype，T4 必须走 float16）
import sys, torch
print('Python', sys.version.split()[0])
print('torch ', torch.__version__)
assert torch.cuda.is_available(), "没有 GPU！请先：修改 → 笔记本设置 → 硬件加速器 → T4 GPU → 保存"
p = torch.cuda.get_device_properties(0)
print('GPU   :', p.name, '| VRAM', round(p.total_memory / 1e9, 1), 'GB | sm_%d%d' % (p.major, p.minor))
bf16_ok = p.major >= 8
print('bf16 硬件支持:', bf16_ok)
DTYPE = 'bfloat16' if bf16_ok else 'float16'
print('>>> 本会话使用 dtype =', DTYPE)

In [ ]:
# ── 1. 装依赖：只装训练必需项，绕开 numpy==1.26.4 的 Python3.13 死结
#    清单由 requirements.txt + 逐个 grep 训练脚本 import 核对得出
!pip -q install "numpy>=2.1" "datasets" "transformers" "einops" "rich" "huggingface_hub" \
    "jsonlines" "datasketch" "simhash" "psutil" "jieba" "nltk" "scikit_learn" "trl" "wandb" "swanlab"

import numpy, datasets, transformers
print('numpy', numpy.__version__, '| datasets', datasets.__version__, '| transformers', transformers.__version__)

# 逐个验证训练脚本真正 import 的包（漏装会跑到一半才崩）
for _m in ['rich', 'jsonlines', 'psutil', 'einops', 'wandb', 'trl']:
    try:
        __import__(_m); print('  ok  ', _m)
    except Exception as _e:
        print('  MISS', _m, _e)

In [ ]:
# ── 2. 挂 Drive + 把项目放 Drive
#    原因：train_pretrain.py 里 checkpoint 路径硬编码为 '../checkpoints'，
#    项目放 Drive 后它天然持久化，省掉软链，实例释放也不丢权重
from google.colab import drive
drive.mount('/content/drive')
import os

PROJ = '/content/drive/MyDrive/minimind'
!mkdir -p /content/drive/MyDrive/minimind
%cd /content/drive/MyDrive/minimind
if not os.path.exists(PROJ + '/trainer'):
    !git clone -q --depth 1 https://github.com/Ezra-Maker-MAX/minimind.git /content/mm_tmp
    !cp -r /content/mm_tmp/. /content/drive/MyDrive/minimind/
    !rm -rf /content/mm_tmp
print('项目目录:', os.getcwd())
!ls

---
## 数据集：分阶段、分层存放

真实体积（实测，非 README 标称）：

| 阶段 | 文件 | 体积 | 免费 Drive 15GB |
|---|---|---|---|
| **P1 基座** | `pretrain_t2t_mini` + `sft_t2t_mini` | **2.98 GB** | ✅ |
| **P2 对齐** | `dpo` + `rlaif` + `agent_rl` + `agent_rl_math` | **178 MB** | ✅ |
| **P3 定制** | `lora_medical` + `lora_identity` + `lora_exam` | **59 MB** | ✅ |
| 完整版 | `pretrain_t2t` + `sft_t2t` | **22.4 GB** | ❌ 装不下 |

流程：**Drive 持久化**（只下一次）→ **每次会话 copy 到 `/content`**（训练读本地盘，Drive 走 FUSE 会拖慢 dataloader）。

> 💡 **性价比提示**：P2 + P3 合计不到 300MB，单次会话就能跑完并拿到结果；
> P1 需 6h+ 且随时可能被回收。建议先做 P2/P3。

In [ ]:
# ── 3. 一次性：把数据集灌进 Google Drive（已存在则跳过，可放心重复运行）
import os, time
from huggingface_hub import hf_hub_download

DD = '/content/drive/MyDrive/minimind/dataset'   # 持久层
LD = '/content/minimind/dataset'                 # 每次会话的工作层（VM 本地盘）
os.makedirs(DD, exist_ok=True); os.makedirs(LD, exist_ok=True)

def free_gb(p):
    s = os.statvfs(p); return s.f_bavail * s.f_frsize / 1e9

print('Drive 剩余 %.1f GB | 本地盘剩余 %.1f GB' % (free_gb('/content/drive'), free_gb('/content')))

# 按阶段选数据；首次建议只跑 P2/P3（体积小、见效快）
STAGE = 'p2'          # mini / p2 / p3 / full
GROUPS = {
    'mini': ['pretrain_t2t_mini.jsonl', 'sft_t2t_mini.jsonl'],
    'p2':   ['dpo.jsonl', 'rlaif.jsonl', 'agent_rl.jsonl', 'agent_rl_math.jsonl'],
    'p3':   ['lora_medical.jsonl', 'lora_identity.jsonl', 'lora_exam.jsonl'],
    'full': ['pretrain_t2t.jsonl', 'sft_t2t.jsonl'],
}
NEED = {'mini': 2.98, 'p2': 0.18, 'p3': 0.06, 'full': 22.4}
FILES = GROUPS[STAGE]
if free_gb('/content/drive') < NEED[STAGE] + 1:
    raise SystemExit('Drive 空间不足（需要 %.2f GB），改用更小的阶段组合' % NEED[STAGE])

for fn in FILES:
    dst = os.path.join(DD, fn)
    if os.path.exists(dst) and os.path.getsize(dst) > 1024:
        print('[已在 Drive]', fn, round(os.path.getsize(dst)/1e6, 1), 'MB'); continue
    t0 = time.time()
    p = hf_hub_download(repo_id='jingyaogong/minimind_dataset', filename=fn,
                        repo_type='dataset', local_dir=DD)
    sz = os.path.getsize(p) / 1e6
    print('[下载] %s %.1f MB  用时 %.0fs  (%.1f MB/s)' % (fn, sz, time.time()-t0, sz/max(time.time()-t0,1)))

In [ ]:
# ── 4. 每次会话：Drive → /content 本地盘（copy2 保留 mtime，便于 datasets 复用 arrow 缓存）
import os, shutil, time
for fn in FILES:                      # 沿用上一格选定的 STAGE
    src, dst = os.path.join(DD, fn), os.path.join(LD, fn)
    if os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src):
        print('[本地已有]', fn, round(os.path.getsize(dst)/1e6, 1), 'MB'); continue
    t0 = time.time(); shutil.copy2(src, dst); dt = time.time() - t0
    sz = os.path.getsize(dst) / 1e6
    print('[同步] %s %.1f MB  用时 %.0fs  (%.0f MB/s)' % (fn, sz, dt, sz/max(dt,1)))

In [ ]:
# ── 5. 冒烟测试：确认模型能实例化、参数量正确、能跑完整一步（先别烧 GPU 时长）
import sys, os, json
PROJ = '/content/drive/MyDrive/minimind'
sys.path.insert(0, PROJ)
from model.model_minimind import MiniMindConfig, MiniMindForCausalLM
from transformers import AutoTokenizer

cfg = MiniMindConfig()
model = MiniMindForCausalLM(cfg)
n = sum(q.numel() for q in model.parameters())
print('参数量: %.1fM  (README 标称 64M)' % (n / 1e6))

tok = AutoTokenizer.from_pretrained(PROJ + '/model')
print('tokenizer 词表:', len(tok))

# 造 200 行假数据走通训练流程
os.makedirs(PROJ + '/dataset', exist_ok=True)
with open(PROJ + '/dataset/smoke.jsonl', 'w', encoding='utf-8') as f:
    for i in range(200):
        f.write(json.dumps({'text': '今天天气很好，我们去公园散步吧。这是第%d条测试数据。' % i},
                           ensure_ascii=False) + '\n')
print('smoke.jsonl 就绪')

In [ ]:
# ── 6. 冒烟训练：真跑几步，验证 forward/backward/保存 全链路
!cd /content/drive/MyDrive/minimind/trainer && python train_pretrain.py \
    --epochs 1 --batch_size 8 --accumulation_steps 1 --num_workers 1 \
    --dtype {DTYPE} --data_path ../dataset/smoke.jsonl \
    --log_interval 5 --save_interval 1000

---
## P1 基座训练（长线，可分多次会话）

冒烟通过后再跑下面两格。**注意时长**：T4 约为 3090 的 40~50% 算力，
mini 数据 1 epoch 的 pretrain 预计 **2.5~3 小时**，SFT 还要更久。

**提速选项**：下面命令已带 `--use_compile 1`。`train_pretrain.py:106` / `train_full_sft.py:107`
都有这个开关（默认 0，上游文档未提）。开启后首次 step 有 2~5 分钟编译开销，之后明显加速。
若遇到 compile 报错，去掉该参数即可回退。

Colab 免费层会话随时可能被回收，所以：
- `--from_resume 1`：断了重跑会自动接上
- 权重和检查点都落在 Drive，实例释放不丢

In [ ]:
# ── 7. 预训练（minimind-3, 64M）—— 数据读本地盘 /content/minimind/dataset
!cd /content/drive/MyDrive/minimind/trainer && python train_pretrain.py \
    --epochs 1 --batch_size 16 --accumulation_steps 8 --num_workers 2 \
    --dtype {DTYPE} --use_compile 1 \
    --data_path /content/minimind/dataset/pretrain_t2t_mini.jsonl \
    --log_interval 50 --save_interval 500 --from_resume 1

In [ ]:
# ── 8. 监督微调（基于上一步的 pretrain 权重）
!cd /content/drive/MyDrive/minimind/trainer && python train_full_sft.py \
    --epochs 1 --batch_size 8 --accumulation_steps 2 --num_workers 2 \
    --dtype {DTYPE} --use_compile 1 \
    --data_path /content/minimind/dataset/sft_t2t_mini.jsonl \
    --log_interval 50 --save_interval 500 --from_resume 1

---
## P2 对齐（性价比最高，单次会话可完成）

**DPO 不需要额外模型**，直接可跑。

⚠️ **GRPO / Agent 有隐藏门槛**：`train_grpo.py` 和 `train_agent.py` 依赖 Reward Model
（`trainer_utils.LMForRewardModel`），默认路径 `../../internlm2-1_8b-reward`，
需额外下载 `internlm/internlm2-1_8b-reward`（约 3.6GB）。免费 Drive 下与 mini 数据共存会吃紧。
**下面先只跑不需要 reward model 的 DPO。**

In [ ]:
# ── 10. DPO 对齐（基于 full_sft 权重，无需 reward model）
!cd /content/drive/MyDrive/minimind/trainer && python train_dpo.py \
    --epochs 1 --batch_size 4 --accumulation_steps 2 --num_workers 2 \
    --dtype {DTYPE} \
    --data_path /content/minimind/dataset/dpo.jsonl \
    --from_weight full_sft --from_resume 1

---
## P3 定制（LoRA，最快见效）

LoRA 只训练低秩适配器，数据量小、耗时短。

| 数据集 | `--lora_name` | 用途 |
|---|---|---|
| `lora_medical.jsonl` 34MB | `lora_medical` | 医疗问答风格 |
| `lora_identity.jsonl` 22.8KB | `lora_identity` | 身份自我认知 |

In [ ]:
# ── 11. LoRA 医疗
!cd /content/drive/MyDrive/minimind/trainer && python train_lora.py \
    --lora_name lora_medical --epochs 3 --batch_size 16 --num_workers 2 \
    --dtype {DTYPE} \
    --data_path /content/minimind/dataset/lora_medical.jsonl \
    --from_weight full_sft --from_resume 1

In [ ]:
# ── 12. LoRA 身份（数据仅 22.8KB，秒级跑完）
!cd /content/drive/MyDrive/minimind/trainer && python train_lora.py \
    --lora_name lora_identity --epochs 10 --batch_size 16 --num_workers 2 \
    --dtype {DTYPE} \
    --data_path /content/minimind/dataset/lora_identity.jsonl \
    --from_weight full_sft --from_resume 1

In [ ]:
# ── 13. 推理验证（权重直接落在 Drive 的项目里，不会丢）
!ls -la /content/drive/MyDrive/minimind/out
!cd /content/drive/MyDrive/minimind && python eval_llm.py --weight full_sft --load_from out